## Running the model

To run the accompanying .prm files simulating the dynamics of a subducting oceanic plate under an overriding plate, execute the following lines with ASPECT:

In [ ]:
%%capture
! aspect prm_files/subduction.prm

> In this above cells, it is assumed that the ASPECT directory is installed system-wide. If not, modify the ASPECT executable to the location where it is installed.

In [ ]:
# Load the relevant libraries
import pyvista as pv
import glob
from IPython.display import HTML

pv.global_theme.allow_empty_mesh = True

UsageError: Cell magic `%%` not found.


In [ ]:
def plot_deformation(output_dir, field, gif_file, cmap, clim):
    '''
    Process a series of VTU files in the given output directory, returning the 'field'
    data for plotting.
    Inputs:
    -----------
    output_dir : str
        Path to the folder containing solution VTU files (expected in solution/solution-*.vtu)
    field : str
        Name of the field we want to look at in the solution
    label : str
        Label for the field being plotted
    cmap  : str
        Colorscale to use for plotting the field
    clim : list
        Color limits for the plot (e.g., [0, 1])
    '''

    solution_file_names = sorted(glob.glob(output_dir + 'solution/solution-*.pvtu'))

    mesh = pv.read(solution_file_names[0])

    plotter = pv.Plotter()

    # Create the first frame
    plotter.add_mesh(mesh, scalars=field, show_scalar_bar=False)
    plotter.add_axes()

    plotter.hide_axes()

    # We know that the model is XY plane so the following
    # lines just sets up the camera
    plotter.view_xy()
    plotter.camera.roll = 0
    plotter.camera.parallel_projection = True
    target_point = (5000e3, 2500e3, 0)
    plotter.camera.focal_point = target_point
    plotter.camera.parallel_scale = 800e3

    sargs_1 = dict(height=0.08, vertical=False, position_x=0.5, position_y=1.5, n_labels=7,
               title='Temperature (K)')

    plotter.open_gif(gif_file, fps=4)

    # Update function for each frame
    def update(frame):
        frame_int = int(frame)
        new_mesh = pv.read(solution_file_names[frame_int])

        plotter.clear_actors()

        plotter.add_mesh(new_mesh, scalars=field, cmap=cmap, scalar_bar_args=sargs_1, \
                         clim=clim)

         # Add a slice at y = 2240 km (or 660 km from the top)
        slice_position = 2240e3
        slice_mesh = mesh.slice(normal='y', origin=(0, slice_position, 0))
        plotter.add_mesh(slice_mesh, color='white', show_scalar_bar=False, name='mesh', line_width=5)

    n_frames = len(solution_file_names)

    for i in range(n_frames):
        update(i)
        plotter.write_frame()

    plotter.close()

In [ ]:
plot_deformation('subduction/', 'T', 'images/subduction.gif', 'RdBu_r', [273, 2000])

In [ ]:
# Now, you would have the gifs saved in the current directory
# this cell visualizes the model outputs side by side for
# comparison

HTML(f"""
<div style="display: flex; justify-content: left; align-items: center; gap: 10px;">
    <img src="{'images/subduction.gif'}" width="500">
    Figure 1: Zoomed in view around subducting slab. The white line indicates the top of the lower mantle.
</div>
""")

### Exercises

- An older overriding plate is stronger and resists to bending, which slows down trench retreat and forces the subducting slab to bend more steeply. Try changing the overriding plate age from 20 Myr to 100 Myr and observe how the slab shape changes.

- A small viscosity jump allows slab to sink into the lower mantle, while a large jump slows the slab down and causes it to flatten or fold at the boundary. Try a stronger viscosity jump from the upper mantle to lower mantle by increasing the `diffusion creep prefactor` (last value in the list, that corresponds to the lower mantle).

- A thicker or weaker layer above the slab allows the plates to slide past each other more easily, which increases trench retreat and changes the overall subduction. Change the layer's properties to see how it affects the subduction style.